In [ ]:
# Install dependencies
!pip install timm diffusers tqdm accelerate


In [ ]:
# Install PyTorch with CUDA 12.8
# !pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio #--index-url https://pypi.tuna.tsinghua.edu.cn/whl/cu128


In [ ]:
# can ignore this
!pip install flash-attn --no-build-isolation --no-cache-dir


In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# IMPORTANT: This notebook is designed for YOUR CUSTOM DiT fork with x2 modifications.
# 
# Option 1: If you've pushed your code to GitHub, replace the URL below:
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git
# %cd YOUR_REPO
#
# Option 2: For now, we clone the base DiT repo and overwrite with your custom files:
%cd /home/ubuntu/
!git clone -b optimze-train-timing https://github.com/mrdjango/DuoDiT.git
%cd DuoDiT
!git checkout optimze-train-timing
!git pull

# The next cells will overwrite models.py (with x2 modifications) and add new scripts


In [ ]:
# Download ImageNet-1k validation data from Hugging Face.
# Source: https://huggingface.co/datasets/mlx-vision/imagenet-1k
# The 6.7 GB val.zip contains 50,000 images already grouped by WNID class.

import json
import shutil
import zipfile
from pathlib import Path

from huggingface_hub import hf_hub_download
from torchvision.datasets import ImageFolder

HF_REPO = 'mlx-vision/imagenet-1k'
DOWNLOAD_DIR = Path('imagenet_hf_download')
DATASET_DIR = Path('imagenet_val_organized')
COMPLETE_MARKER = DATASET_DIR / '.complete'

# The class map defines the exact ImageNet/DiT class order: 0..999 -> WNID.
class_map_file = hf_hub_download(
    repo_id=HF_REPO,
    repo_type='dataset',
    filename='class_map.json',
    local_dir=DOWNLOAD_DIR,
)
with open(class_map_file) as f:
    class_id_to_wnid = {int(class_id): wnid for class_id, wnid in json.load(f).items()}

assert set(class_id_to_wnid) == set(range(1000))
expected_class_to_idx = {wnid: class_id for class_id, wnid in class_id_to_wnid.items()}

existing_images = sum(1 for path in DATASET_DIR.glob('*/*') if path.is_file())
dataset_ready = COMPLETE_MARKER.exists() and existing_images == 50000

if not dataset_ready:
    print('Downloading ImageNet validation set from Hugging Face (~6.7 GB)...')
    archive_file = hf_hub_download(
        repo_id=HF_REPO,
        repo_type='dataset',
        filename='val.zip',
        local_dir=DOWNLOAD_DIR,
    )

    extract_dir = Path('imagenet_val_extracted')
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir()

    print('Extracting 50,000 validation images...')
    with zipfile.ZipFile(archive_file) as archive:
        archive.extractall(extract_dir)

    # val.zip stores the 1,000 WNID folders either directly or inside val/.
    expected_wnids = set(expected_class_to_idx)
    candidates = [extract_dir] + [path for path in extract_dir.iterdir() if path.is_dir()]
    source_dir = next(
        (
            candidate for candidate in candidates
            if {path.name for path in candidate.iterdir() if path.is_dir()} == expected_wnids
        ),
        None,
    )
    if source_dir is None:
        raise RuntimeError('Could not find the 1,000 WNID class folders in val.zip.')

    if DATASET_DIR.exists():
        shutil.rmtree(DATASET_DIR)
    shutil.move(str(source_dir), str(DATASET_DIR))
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    COMPLETE_MARKER.write_text('50000\n')
else:
    print('Complete ImageNet validation dataset already exists; skipping download.')

# ImageFolder sorts WNIDs in the same order used by pretrained ImageNet DiT.
dataset_check = ImageFolder(DATASET_DIR)
assert dataset_check.class_to_idx == expected_class_to_idx
assert len(dataset_check) == 50000
assert set(dataset_check.targets) == set(range(1000))

print(f'Ready: {len(dataset_check):,} images, {len(dataset_check.classes)} classes')
print(f'DiT labels: {class_id_to_wnid[0]} -> 0, {class_id_to_wnid[999]} -> 999')
print('Training path: ./imagenet_val_organized')


In [ ]:
# Custom results directory
!torchrun --nnodes=1 --nproc_per_node=1 --master_addr=127.0.0.1 --master_port=29500 train_x2_finetune.py \
    --model DiT-XL/2 \
    --data-path ./imagenet_val_organized \
    --results-dir ./results/checkpoints \
    --pretrained-ckpt DiT-XL-2-256x256.pt \
    --epochs 400 \
    --classes $(python3 -c "print(' '.join(map(str, range(1000))))") \
    --global-batch-size 50 \
    --log-every 100 \
    --ckpt-every 50000 \
    # --num-workers 2 \
    # --prefetch-factor 2 \
    # --debug-progress \
    # --progress-leave


In [ ]:
!python3 -c "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available(), torch.cuda.get_device_name(0))"


In [ ]:
import os
import shutil
import socket
import platform
import subprocess
import psutil
import torch

print("=" * 60)
print("SYSTEM INFORMATION")
print("=" * 60)

# Host / OS
print(f"Hostname: {socket.gethostname()}")
print(f"Platform: {platform.platform()}")
print(f"Python: {platform.python_version()}")

print("\n" + "=" * 60)
print("CPU INFORMATION")
print("=" * 60)

print(f"Physical cores: {psutil.cpu_count(logical=False)}")
print(f"Logical cores: {psutil.cpu_count(logical=True)}")
print(f"CPU usage: {psutil.cpu_percent(interval=1)}%")

try:
    cpu_name = subprocess.check_output(
        "lscpu | grep 'Model name'",
        shell=True
    ).decode().strip()
    print(cpu_name)
except:
    pass

print("\n" + "=" * 60)
print("RAM INFORMATION")
print("=" * 60)

ram = psutil.virtual_memory()

print(f"Total RAM: {ram.total / (1024**3):.2f} GB")
print(f"Available RAM: {ram.available / (1024**3):.2f} GB")
print(f"Used RAM: {ram.used / (1024**3):.2f} GB")
print(f"RAM Usage: {ram.percent}%")

print("\n" + "=" * 60)
print("DISK INFORMATION")
print("=" * 60)

disk = shutil.disk_usage("/")

print(f"Total Disk: {disk.total / (1024**3):.2f} GB")
print(f"Used Disk: {disk.used / (1024**3):.2f} GB")
print(f"Free Disk: {disk.free / (1024**3):.2f} GB")

print("\n" + "=" * 60)
print("GPU INFORMATION")
print("=" * 60)

print(f"Torch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f"GPU count: {gpu_count}")

    for i in range(gpu_count):
        props = torch.cuda.get_device_properties(i)

        total_mem = props.total_memory / (1024**3)

        allocated = torch.cuda.memory_allocated(i) / (1024**3)
        reserved = torch.cuda.memory_reserved(i) / (1024**3)

        print("\n" + "-" * 40)
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"VRAM Total: {total_mem:.2f} GB")
        print(f"VRAM Allocated: {allocated:.2f} GB")
        print(f"VRAM Reserved: {reserved:.2f} GB")
        print(f"Compute Capability: {props.major}.{props.minor}")
        print(f"Multiprocessors: {props.multi_processor_count}")

print("\n" + "=" * 60)
print("NVIDIA-SMI")
print("=" * 60)

try:
    os.system("nvidia-smi")
except:
    print("nvidia-smi not available")
